In [4]:
import os
import hashlib

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from hash import generar_hash

In [5]:
load_dotenv()

SALT = os.getenv("HASH_SALT")
if not SALT:
    raise RuntimeError("No se encontró la variable de entorno HASH_SALT")

# Postgres
PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB   = os.getenv("PG_DB")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

# MySQL
MY_HOST = os.getenv("MY_HOST")
MY_PORT = os.getenv("MY_PORT", "3306")
MY_DB   = os.getenv("MY_DB")
MY_USER = os.getenv("MY_USER")
MY_PASSWORD = os.getenv("MY_PASSWORD")


### Limpieza CSV

In [6]:
np.random.seed(42)

df = pd.read_csv("data/df_general_errores.csv")

# Eliminar filas totalmente vacías
df = df.dropna(how="all")

# Limpiar nombre y apellido
for col in ["nombre", "apellido"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .replace({"nan": np.nan, "NaN": np.nan})
            .str.strip()
        )

# Mantener solo registros con nombre y apellido válidos
if {"nombre", "apellido"}.issubset(df.columns):
    mask_valid = df["nombre"].notna() & df["apellido"].notna()
    mask_valid &= df["nombre"] != ""
    mask_valid &= df["apellido"] != ""
    df = df[mask_valid].copy()

# Quitar duplicados globales
df = df.drop_duplicates().reset_index(drop=True)

# Definir columnas numéricas (solo si existen)
numeric_cols = ["longitud", "latitud", "distancia", "semestre", "anio", "edad"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# El resto se consideran texto
text_cols = [c for c in df.columns if c not in numeric_cols]

# --------- Tratamiento de numéricas: outliers + imputación media ---------

for col in numeric_cols:
    col_num = pd.to_numeric(df[col], errors="coerce")

    not_na = col_num.dropna()
    if len(not_na) >= 5:
        q1 = not_na.quantile(0.25)
        q3 = not_na.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = (col_num < lower) | (col_num > upper)
        col_num[outliers] = np.nan

    mean_val = col_num.mean()
    col_num = col_num.fillna(mean_val)

    df[col] = col_num

# Redondeo para variables que deben ser enteras
for col in ["semestre", "anio", "edad"]:
    if col in df.columns:
        df[col] = df[col].round().astype(int)

# --------- Tratamiento de texto: reemplazo de valores corruptos ---------

invalid_tokens = {
    "###", "!ERROR!", "###CORRUPT###", "@@@", "<NULL>",
    "INVALID", "BROKEN", "{BAD}", "<script>",
    "ERR", "???", "BAD", "∞∞∞", "±§¶", "ERR!!", "$$$",
    "<NULL>", "C0RRUPT3D", "[DAMAGED]",
    "~~~~", "XXX", "BAD"
}

def is_invalid_text(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == "":
        return True
    if s in invalid_tokens:
        return True
    if len(s) > 80:
        return True
    if set(s) <= set("#@!<>={}[]±§¶∞") and len(s) <= 10:
        return True
    return False

for col in text_cols:
    serie = df[col].astype(object)

    valid_mask = ~serie.map(is_invalid_text)
    invalid_mask = ~valid_mask

    # Si todo es inválido, no tocamos la columna
    if valid_mask.sum() == 0:
        df[col] = serie
        continue

    valid_vals = serie[valid_mask].astype(str)
    dist = valid_vals.value_counts(normalize=True)

    valores = dist.index.to_numpy()
    probs = dist.values

    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        relleno = np.random.choice(valores, size=n_invalid, p=probs)
        serie.loc[invalid_mask] = relleno

    df[col] = serie

# Quitar duplicados por nombre + apellido
if {"nombre", "apellido"}.issubset(df.columns):
    df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

df_general = df.reset_index(drop=True)



In [7]:
# Generar identificador único
def generar_hash(nombre, apellido):
    base = f"{nombre}_{apellido}"
    return hashlib.md5(base.encode("utf-8")).hexdigest()

df_general["identifier"] = df_general.apply(
    lambda row: generar_hash(row["nombre"], row["apellido"]),
    axis=1
)

### Limpieza xlsx

In [8]:

df = pd.read_excel("data/df_bienestar_errores.xlsx")
print(df.columns)

np.random.seed(42)

# Eliminar filas totalmente vacías
df = df.dropna(how="all")

# Limpiar nombre y apellido
for col in ["nombre", "apellido"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .replace({"nan": np.nan, "NaN": np.nan})
            .str.strip()
        )

# Mantener solo registros con nombre y apellido válidos
if {"nombre", "apellido"}.issubset(df.columns):
    mask_valid = df["nombre"].notna() & df["apellido"].notna()
    mask_valid &= df["nombre"] != ""
    mask_valid &= df["apellido"] != ""
    df = df[mask_valid].copy()

# Quitar duplicados por nombre + apellido
df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

# Columnas numéricas (solo si existen)
numeric_cols = [
    "distancia",
    "promedio_académico",
    "viajes_origen",
    "indice_bienestar",
    "distancia_universidad",
    "horas_sueño",
    "tiempo_traslado_min",
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

# El resto se consideran texto
text_cols = [c for c in df.columns if c not in numeric_cols]

# --- Tratamiento numérico: outliers + media ---

for col in numeric_cols:
    col_num = pd.to_numeric(df[col], errors="coerce")

    not_na = col_num.dropna()
    if len(not_na) >= 5:
        q1 = not_na.quantile(0.25)
        q3 = not_na.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = (col_num < lower) | (col_num > upper)
        col_num[outliers] = np.nan

    mean_val = col_num.mean()
    col_num = col_num.fillna(mean_val)

    df[col] = col_num

# Redondear viajes_origen a entero
if "viajes_origen" in df.columns:
    df["viajes_origen"] = df["viajes_origen"].round().astype(int)

# --- Reutilizamos is_invalid_text del código anterior ---

def is_invalid_text(v):
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == "":
        return True
    invalid_tokens = {
        "###", "!ERROR!", "###CORRUPT###", "@@@", "<NULL>",
        "INVALID", "BROKEN", "{BAD}", "<script>",
        "ERR", "???", "BAD", "∞∞∞", "±§¶", "ERR!!", "$$$",
        "<NULL>", "C0RRUPT3D", "[DAMAGED]",
        "~~~~", "XXX", "BAD"
    }
    if s in invalid_tokens:
        return True
    if len(s) > 80:
        return True
    if set(s) <= set("#@!<>={}[]±§¶∞") and len(s) <= 10:
        return True
    return False

# --- Tratamiento de texto: reemplazo de valores corruptos ---

for col in text_cols:
    serie = df[col].astype(object)

    valid_mask = ~serie.map(is_invalid_text)
    invalid_mask = ~valid_mask

    if valid_mask.sum() == 0:
        df[col] = serie
        continue

    valid_vals = serie[valid_mask].astype(str)
    dist = valid_vals.value_counts(normalize=True)

    valores = dist.index.to_numpy()
    probs = dist.values

    n_invalid = invalid_mask.sum()
    if n_invalid > 0:
        relleno = np.random.choice(valores, size=n_invalid, p=probs)
        serie.loc[invalid_mask] = relleno

    df[col] = serie

# Quitar duplicados por nombre + apellido otra vez por si se generaron
df = df.drop_duplicates(subset=["nombre", "apellido"]).reset_index(drop=True)

# (Opcional) generar el mismo identifier para poder hacer joins
df["identifier"] = df.apply(
    lambda row: generar_hash(row["nombre"], row["apellido"]),
    axis=1
)

df_bienestar = df.reset_index(drop=True)


Index(['nombre', 'apellido', 'ciudad', 'provincia', 'distancia',
       'promedio_académico', 'viajes_origen', 'indice_bienestar',
       'origen_bienestar', 'distancia_universidad', 'horas_sueño',
       'tiempo_traslado_min'],
      dtype='object')


In [9]:

from sqlalchemy import create_engine
# MySQL
mysql_engine = create_engine(
    f"mysql+pymysql://{MY_USER}:{MY_PASSWORD}@{MY_HOST}:{MY_PORT}/{MY_DB}"
)

# Postgres
pg_engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)


# Estudiante desde MySQL
df_mysql_estudiante = pd.read_sql("SELECT * FROM estudiante", mysql_engine)
df_mysql_estudiante = df_mysql_estudiante.drop_duplicates(subset=["nombre", "apellido"])



In [10]:


# ---- dim_universidad ----
dim_universidad = (
    df_general[["universidad", "carrera"]]
    .drop_duplicates()
    .sort_values(["universidad", "carrera"])
    .reset_index(drop=True)
)
dim_universidad["id_universidad"] = np.arange(1, len(dim_universidad) + 1)

# ---- dim_periodo ----
dim_periodo = (
    df_general[["semestre", "anio", "periodo"]]
    .drop_duplicates()
    .sort_values(["semestre", "anio", "periodo"])
    .reset_index(drop=True)
)
dim_periodo["id_periodo"] = np.arange(1, len(dim_periodo) + 1)


# ---- dim_estudiante ----
# Usamos el identifier de df_general y cruzamos con MySQL (becado)

dim_estudiante = (
    df_general.merge(
        df_mysql_estudiante[["nombre", "apellido", "es_becado"]],
        on=["nombre", "apellido"],
        how="inner"
    )
)

dim_estudiante = (
    dim_estudiante[["identifier", "edad", "genero", "modalidad", "es_becado"]]
    .drop_duplicates()
    .rename(columns={
        "identifier": "id_estudiante",
        "es_becado": "becado"
    })
    .reset_index(drop=True)
)

# ---- dim_origen ----
dim_origen = (
    df_general[["ciudad", "provincia"]]
    .drop_duplicates()
    .sort_values(["ciudad", "provincia"])
    .reset_index(drop=True)
)
dim_origen["id_origen"] = np.arange(1, len(dim_origen) + 1)




In [11]:

df_pg_categoria = pd.read_sql(
    "SELECT DISTINCT categoria FROM gastos WHERE categoria IS NOT NULL",
    pg_engine
)

dim_categoria_gasto = (
    df_pg_categoria[["categoria"]]
    .drop_duplicates()
    .sort_values("categoria")
    .reset_index(drop=True)
)

dim_categoria_gasto["id_categoria_gasto"] = np.arange(1, len(dim_categoria_gasto) + 1)  
dim_categoria_gasto = dim_categoria_gasto[["id_categoria_gasto", "categoria"]]


In [12]:
df_gastos = pd.read_sql("""
    SELECT 
        g.egreso_cantidad AS monto_gasto,
        g.categoria,
        e.nombre,
        e.apellido
    FROM gastos g
    JOIN estudiante e ON e.id_estudiante = g.id_estudiante
""", pg_engine)
fact_gasto = df_gastos.merge(
    dim_categoria_gasto,
    on="categoria",
    how="inner"
)



In [13]:


general_keys = (
    df_general[
        [
            "nombre",
            "apellido",
            "universidad",
            "carrera",
            "semestre",
            "anio",
            "periodo",
            "ciudad",
            "provincia",
        ]
    ]
    .drop_duplicates()
)

# mapa estudiante: identifier -> id_estudiante
map_estudiante = (
    df_general[["identifier", "nombre", "apellido"]]
    .drop_duplicates()
    .rename(columns={"identifier": "id_estudiante"})
)


fact = (
    fact_gasto
    .merge(general_keys, on=["nombre", "apellido"], how="inner")
    .merge(map_estudiante, on=["nombre", "apellido"], how="inner")
    .merge(
        dim_universidad[["id_universidad", "universidad", "carrera"]],
        on=["universidad", "carrera"],
        how="inner",
    )
    # periodo -> id_periodo
    .merge(
        dim_periodo[["id_periodo", "semestre", "anio", "periodo"]],
        on=["semestre", "anio", "periodo"],
        how="inner",
    )
    .merge(
        dim_origen[["id_origen", "ciudad", "provincia"]],
        on=["ciudad", "provincia"],
        how="inner",
    )
)

# ================== QUEDARSE SOLO CON FKs + MEDIDA ==================

fact_gasto_modelo = fact[
    [
        "id_estudiante",
        "id_periodo",
        "id_universidad",
        "id_categoria_gasto",
        "id_origen",
        "monto_gasto",
    ]
].drop_duplicates().copy()


In [14]:


ids_est = fact_gasto_modelo["id_estudiante"].unique()
ids_per = fact_gasto_modelo["id_periodo"].unique()
ids_uni = fact_gasto_modelo["id_universidad"].unique()
ids_cat = fact_gasto_modelo["id_categoria_gasto"].unique()
ids_ori = fact_gasto_modelo["id_origen"].unique()


dim_estudiante_gasto = dim_estudiante[
    dim_estudiante["id_estudiante"].isin(ids_est)
].reset_index(drop=True)

dim_periodo_gasto = dim_periodo[
    dim_periodo["id_periodo"].isin(ids_per)
].reset_index(drop=True)

dim_universidad_gasto = dim_universidad[
    dim_universidad["id_universidad"].isin(ids_uni)
].reset_index(drop=True)

dim_categoria_gasto_gasto = dim_categoria_gasto[
    dim_categoria_gasto["id_categoria_gasto"].isin(ids_cat)
].reset_index(drop=True)

dim_origen_gasto = dim_origen[
    dim_origen["id_origen"].isin(ids_ori)
].reset_index(drop=True)


In [15]:
dim_estudiante=dim_estudiante_gasto
dim_periodo=dim_periodo_gasto
dim_universidad=dim_universidad_gasto
dim_categoria_gasto=dim_categoria_gasto_gasto
dim_origen=dim_origen_gasto

# Cargar datos

In [16]:
import os
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv()

CH_HOST = os.getenv("CH_HOST", "localhost")
CH_PORT = int(os.getenv("CH_PORT", "8123"))
CH_USER = os.getenv("CH_USER", "admin")
CH_PASSWORD = os.getenv("CH_PASSWORD", "admin")

CH_DB = "dm_gastos"

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
)


In [17]:
def insert_df(client, table, df):
    cols = list(df.columns)
    data = [tuple(row) for row in df.itertuples(index=False, name=None)]
    client.insert(table, data, column_names=cols)

CH_DM_GASTOS = "dm_gastos"

# ==== DIMENSIONES ====
insert_df(client, f"{CH_DM_GASTOS}.dim_estudiante", dim_estudiante)
insert_df(client, f"{CH_DM_GASTOS}.dim_universidad", dim_universidad)
insert_df(client, f"{CH_DM_GASTOS}.dim_periodo", dim_periodo)
insert_df(client, f"{CH_DM_GASTOS}.dim_categoria_gasto", dim_categoria_gasto)
insert_df(client, f"{CH_DM_GASTOS}.dim_origen", dim_origen)



In [20]:
insert_df(client, f"{CH_DM_GASTOS}.fact_gasto", fact_gasto_modelo)

### Resultado

Después de este proceso, al final quedo un 54% de los datos perfectos para un análisis